# IceCache Demo — Synthetic Long-Context Inference

**IceCache** offloads the KV cache to CPU and uses a Multi-level DCI (Dynamic Continuous Indexing) index to retrieve the most relevant KV pages back to GPU during decoding, enabling long-context inference with a small GPU memory footprint.

This notebook demonstrates IceCache on a **passkey retrieval task** with a **20k-token** synthetic sequence using .

> **Runtime:** GPU (A100 or better recommended)  
> **HuggingFace token:** required for Llama 3.1 — set it below.

## 1. Install dependencies

In [ ]:
# Install PyTorch (already present in Colab, but pin to a compatible version)
import torch
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Clone IceCache
!git clone https://github.com/yuzhenmao/IceCache.git --recursive
%cd IceCache

In [ ]:
# Python dependencies
!pip install -q -r IceCache/requirements.txt
!pip install -q flash_attn==2.6.1 --no-build-isolation

In [ ]:
# IceCache Python package
%cd IceCache/source
!pip install -q -e . --no-build-isolation
%cd ../..

In [ ]:
# Build & install the DCI C extension
%cd Multi-level-DCI
!python3 setup.py install
%cd ..

In [ ]:
# Verify
from dciknn import DCI
import icecache
print("DCI and IceCache imported successfully.")

## 2. Load the model

In [ ]:
import os

# Paste your HuggingFace token here (needs access to meta-llama/Llama-3.1-8B-Instruct)
HF_TOKEN = "hf_YOUR_TOKEN_HERE"  # @param {type:"string"}
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = "meta-llama/Llama-3.1-8B-Instruct"
device = torch.device("cuda")

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False, token=HF_TOKEN)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map=device,
    token=HF_TOKEN,
).eval()
print("Model loaded.")

## 3. Enable IceCache

In [ ]:
from icecache import adapter

# IceCache configuration
PAGE_SIZE    = 16   # tokens per KV page
PAGE_BUDGETS = 16   # max pages kept on GPU per layer
N_SINK_PAGES = 2    # always-kept pages at the start (attention sink)
N_WIN_PAGES  = 2    # always-kept pages at the end (local window)

n_max_bytes = 40 * (1 << 28)  # the max total size of all KV pages (GPU + CPU)
n_max_cpu_bytes = 80 * (1 << 28)  # the max total size of all CPU-resident KV pages (including those swapped out from GPU)


adapter.enable_icecache(
    model,
    dtype=torch.float16,
    device=device,
    page_size=PAGE_SIZE,
    page_budgets=PAGE_BUDGETS,
    n_sink_pages=N_SINK_PAGES,
    n_win_pages=N_WIN_PAGES,
    n_max_bytes=n_max_bytes,
    n_max_cpu_bytes=n_max_cpu_bytes,
)
print("IceCache enabled.")
print(f"GPU budget: {PAGE_BUDGETS} pages x {PAGE_SIZE} tokens = {PAGE_BUDGETS * PAGE_SIZE} tokens on GPU")

## 4. Synthetic passkey retrieval at 20k tokens

We hide a random 5-digit passkey inside 20k tokens of filler text and ask the model to retrieve it.
This tests whether IceCache can correctly identify and fetch the relevant KV pages from CPU.

In [ ]:
import numpy as np

def build_passkey_prompt(seq_length_chars: int = 100_000, passkey_loc: float = 0.5, seed: int = 42):
    """Build a passkey-retrieval prompt of approximately seq_length_chars characters."""
    rng = np.random.default_rng(seed)
    passkey = int(rng.integers(10000, 99999))

    filler = ("The grass is green. The sky is blue. The sun is yellow. Here we go. There and back again. " * 5000)

    prefix_len = int(seq_length_chars * passkey_loc)
    suffix_len = seq_length_chars - prefix_len

    prefix = filler[:prefix_len]
    suffix = filler[:suffix_len]

    task = "There is an important info hidden inside a lot of irrelevant text. Find it and memorize them. I will quiz you about the important information there."
    secret = f"The pass key is {passkey}. Remember it. {passkey} is the pass key."
    question = "What is the pass key? The pass key is"

    prompt = "\n".join([task, prefix, secret, suffix, question])
    return prompt, passkey


# ~100k chars ≈ 20k tokens for Llama tokenizer
PROMPT_CHARS = 100_000
PASSKEY_LOC  = 0.5   # place the key at 50% of the sequence

prompt, true_passkey = build_passkey_prompt(PROMPT_CHARS, PASSKEY_LOC)
inputs = tokenizer([prompt], return_tensors="pt").to(device)
seq_len = inputs.input_ids.shape[-1]
print(f"Sequence length: {seq_len:,} tokens")
print(f"True passkey:    {true_passkey}")

## 5. Run inference

In [ ]:
import time

print("Running inference with IceCache...")
t0 = time.time()

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
    )

elapsed = time.time() - t0
generated = tokenizer.decode(output_ids[0, inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()

print(f"Elapsed: {elapsed:.1f}s")
print(f"Model output: {generated}")
print(f"True passkey:  {true_passkey}")
print(f"Correct: {str(true_passkey) in generated}")